In [2]:
import pandas as pd
import numpy as np
import re
import cloudscraper
import time
import math
import json
import seaborn as sns
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from dateutil.parser import parse

from google import genai
from google.genai.types import Schema, Type

In [ ]:
def func_estimate_gemini_flash_cost(text, expected_output_tokens=200, model="flash"):
    """
    Estimate Gemini Flash API cost for a given text input.
    Args:
        text (str): Input text.
        expected_output_tokens (int): Approx. output token count (e.g. summary length).
        model (str): "flash" or "flash-lite"
    Returns:
        dict: tokens + cost breakdown.
    """
    # rough estimate: ~4 characters per token (avg English text)
    input_tokens = math.ceil(len(text) / 4)
    output_tokens = expected_output_tokens

    # USD cost per million tokens (input / output)
    pricing = {
        "flash": {"input": 0.075, "output": 0.30},
        "flash-lite": {"input": 0.10, "output": 0.40},
    }

    if model not in pricing:
        raise ValueError("Model must be 'flash' or 'flash-lite'")

    cost_input = (input_tokens / 1_000_000) * pricing[model]["input"]
    cost_output = (output_tokens / 1_000_000) * pricing[model]["output"]
    total = cost_input + cost_output

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": round(total, 6),
    }

In [3]:
scraper = cloudscraper.create_scraper()  # handles Cloudflare anti-bot
page_url = "https://www.dawn.com/pakistan"
resp = scraper.get(page_url, timeout=15)
soup = BeautifulSoup(resp.text, 'html.parser')

In [6]:
news_stories_all = pd.read_pickle('../data/dawn_news_stories_sep2025.pkl')


In [18]:
scraped_dates = news_stories_all.timestamp.value_counts()
scraped_dates = scraped_dates.loc[lambda x: x>15]
scraped_dates = set(scraped_dates.index.tolist())

In [23]:
start = pd.Timestamp('2025-01-01')
end  =  pd.Timestamp('2025-12-31')
# end = start + pd.offsets.MonthEnd(0)
month_dates = pd.date_range(start=start, end=end, freq='D').strftime('%Y-%m-%d').tolist()

In [26]:
all_stories = []

for date in month_dates:
    scraper = cloudscraper.create_scraper()  # handles Cloudflare anti-bot
    page_url = "https://www.dawn.com/pakistan/{date}/".format(date=date)
    print(page_url)
    resp = scraper.get(page_url, timeout=15)
    soup = BeautifulSoup(resp.text, 'html.parser')

    # Scrape article/story URLs from home page for Pakistan section
    anchors = soup.find_all("a", href=True)
    urls = set()
    for a in anchors:
        href = a["href"].split("#")[0].split("?")[0]  #  strip fragments and query params
        if not href:
            continue
        # absolute or relative URL to dawn
        if href.startswith("http") and "dawn.com" in href:
            candidate = href
        elif href.startswith("/"):
            candidate = urljoin(page_url, href)
        else:
            continue

        # Filter to likely article/story pages: by year pattern or '/news' or '/pakistan' in path
        if re.search(r"/\d{4}/", candidate) or "/news" in candidate or "/pakistan" in candidate:
            urls.add(candidate)

    urls = sorted(urls)
    for u in urls:
        all_stories.append(u)


https://www.dawn.com/pakistan/2025-01-01/
https://www.dawn.com/pakistan/2025-01-02/
https://www.dawn.com/pakistan/2025-01-03/
https://www.dawn.com/pakistan/2025-01-04/
https://www.dawn.com/pakistan/2025-01-05/
https://www.dawn.com/pakistan/2025-01-06/
https://www.dawn.com/pakistan/2025-01-07/
https://www.dawn.com/pakistan/2025-01-08/
https://www.dawn.com/pakistan/2025-01-09/
https://www.dawn.com/pakistan/2025-01-10/
https://www.dawn.com/pakistan/2025-01-11/
https://www.dawn.com/pakistan/2025-01-12/
https://www.dawn.com/pakistan/2025-01-13/
https://www.dawn.com/pakistan/2025-01-14/
https://www.dawn.com/pakistan/2025-01-15/
https://www.dawn.com/pakistan/2025-01-16/
https://www.dawn.com/pakistan/2025-01-17/
https://www.dawn.com/pakistan/2025-01-18/
https://www.dawn.com/pakistan/2025-01-19/
https://www.dawn.com/pakistan/2025-01-20/
https://www.dawn.com/pakistan/2025-01-21/
https://www.dawn.com/pakistan/2025-01-22/
https://www.dawn.com/pakistan/2025-01-23/
https://www.dawn.com/pakistan/2025

In [ ]:
all_stories = list(set(all_stories))
all_stories = [u for u in all_stories if 'news' in u]
all_stories = [u for u in all_stories if 'images.dawn' not in u] #filter out stories by dawn images
pd.DataFrame(all_stories, columns=['url']).to_pickle('../data/dawn_all_story_urls_2025.pkl')

11397

In [35]:
story_data_list = []

In [ ]:
# Scrape content for each story URL
parsed_urls = set([item['url'] for item in story_data_list])
for story in all_stories:
    if story not in parsed_urls:
        for attempt in range(5):
            try:
                time.sleep(1)
                resp = scraper.get(story, timeout=15)
                soup = BeautifulSoup(resp.text, 'html.parser')
                print(story)
                story_title = soup.title.get_text(strip=True)
                story_content = soup.select_one('div.story__content').get_text(separator='\n', strip=True)
                story_timestamp = soup.select_one('span.timestamp--date').get_text(separator='\n', strip=True)
                story_timestamp = parse(story_timestamp)
                story_byline = soup.select_one('a.story__byline__link').get_text()
                story_data = {
                    'url': story,
                    'title': story_title,
                    'content': story_content,
                    'timestamp': story_timestamp,
                    'byline': story_byline
                }
                story_data_list.append(story_data)
                break  # success, move to next story
            except AttributeError as e:
                if attempt < 4:
                    print(f"AttributeError parsing {story}, retrying ({attempt+1}/5)...")
                    time.sleep(1)
                    continue
                else:
                    print(f"Failed to parse {story} after 5 attempts: {e}")
            except Exception as e:
                # non-AttributeError (network, parser, etc.) — report and skip
                print(f"Error fetching/parsing {story}: {e}")
                break
    else:
        print(f"Skipping already parsed URL: {story}")

https://www.dawn.com/news/1911383/india-weighs-plan-to-slash-pakistan-water-supply-with-new-indus-river-project
https://www.dawn.com/news/1892136/police-constable-shot-dead-in-karachis-manghopir
https://www.dawn.com/news/1940735/pakistan-kazakhstan-sign-action-plan-of-cooperation-to-boost-ties
https://www.dawn.com/news/1904954/ihc-orders-govt-to-submit-reply-to-pleas-against-peca-amendments
https://www.dawn.com/news/1923585/former-lesco-chief-dismissed-for-overbilling-corruption
https://www.dawn.com/news/1930023/karachis-dha-some-other-areas-face-24-hour-gas-shutdown-today
https://www.dawn.com/news/1953337/over-4000-test-positive-for-dengue-in-five-days-in-sindh
https://www.dawn.com/news/1885206/situationer-pakistani-blocs-intensify-lobbying-before-trumps-inauguration
https://www.dawn.com/news/1895453/snowfall-avalanches-wreak-havoc-on-astore
https://www.dawn.com/news/1917996/police-official-public-prosecutor-arrested-for-running-honey-trap-racket-in-rawalpindi
https://www.dawn.com/new

In [ ]:
story_data_df = pd.DataFrame(story_data_list)
story_data_df['input_tokens'] = story_data_df['content'].apply(lambda x: func_estimate_gemini_flash_cost(x)['input_tokens'])
story_data_df['token_cost_usd'] = story_data_df['content'].apply(lambda x: func_estimate_gemini_flash_cost(x)['cost_usd'])